# Notebook 09 — Lotka-Volterra Predator-Prey Model

**Module:** 02 — Mathematics for Biology  
**Notebook:** 09 of 12 — Portfolio artifact  
**Prerequisites:** NB07, NB08  
**Time estimate:** 120 minutes

> **Portfolio note:** This notebook produces `portfolio/module02_lotka_volterra_phase_portrait.png`.
> The final figure must be publication-quality and run reproducibly from a clean clone.

---
## Step 1 — Motivation

Why do Canadian lynx and snowshoe hare populations oscillate with a ~10-year period?
Alfred Lotka (1910, 1925) and Vito Volterra (1926) independently derived the same
two-ODE system that predicts exactly this behavior — without invoking stochasticity,
spatial structure, or any mechanism beyond predation. The Lotka-Volterra model is the
foundational example of a two-dimensional ODE system with limit-cycle dynamics, and
understanding it from scratch — equations, fixed points, phase portrait — is the core
deliverable of Module 02.

---
## Step 2 — Intuition

Two feedback loops:
1. **Prey grows → predator grows** (more food → more predators)
2. **Predator grows → prey declines** (more predators → more prey eaten)

The lag between prey peak and predator peak creates sustained oscillations. Neither
species drives the other to extinction — they cycle endlessly in the neutral case.
This is a *conservative system* (unlike the logistic model which has a stable
fixed point).

---
## Step 3 — Biological Background

**The Hudson Bay Company lynx/hare data** (1845–1935): 90 years of pelt-count records
show ~10-year oscillations between lynx and hare. The Lotka-Volterra model reproduces
this period qualitatively without fitting — it emerges from the equations.

**Parameters and their biological meaning:**

| Parameter | Symbol | Biological meaning | Units |
|-----------|--------|-------------------|-------|
| Prey growth rate | $\alpha$ | Birth rate of prey when predators absent | per time |
| Predation rate | $\beta$ | Rate at which predators consume prey | per predator·time |
| Predator death rate | $\gamma$ | Death rate of predators when prey absent | per time |
| Predator growth efficiency | $\delta$ | Conversion of prey to predator births | per prey·time |

**Assumptions (all violated in nature; model still instructive):**
- Unlimited prey growth without predators (no carrying capacity)
- Predator consumption linear in prey density (no saturation)
- No spatial structure, age structure, or seasonal forcing

---
## Step 4 — Mathematical Explanation

**The Lotka-Volterra ODE system:**
$$\frac{dx}{dt} = \alpha x - \beta x y \quad \text{(prey)}$$
$$\frac{dy}{dt} = \delta x y - \gamma y \quad \text{(predator)}$$

**Fixed points** (set both derivatives to zero):
1. $(x^*, y^*) = (0, 0)$ — extinction (unstable)
2. $(x^*, y^*) = (\gamma/\delta,\; \alpha/\beta)$ — coexistence fixed point

**Conservation law (Lyapunov function):**
$$V(x, y) = \delta x - \gamma \ln x + \beta y - \alpha \ln y = \text{const}$$

$V$ is constant along trajectories — the system is conservative. This is why
trajectories are closed loops around the fixed point rather than spiraling in or out.

**Period of oscillation:** approximately $T \approx 2\pi / \sqrt{\alpha \gamma}$ for
small-amplitude oscillations near the fixed point.

---
## Step 5 — Computational Explanation

**Phase portrait construction:**
1. Solve the ODE with `solve_ivp` (and verify with hand-rolled `rk4` from NB08)
2. Plot trajectory in $(x, y)$ space (the phase plane)
3. Add nullclines: curves where $dx/dt = 0$ and $dy/dt = 0$
4. Mark the coexistence fixed point
5. Add a quiver field showing the direction of flow at each point

**Nullclines:**
- $dx/dt = 0$: $x = 0$ or $y = \alpha/\beta$ (horizontal prey nullcline)
- $dy/dt = 0$: $y = 0$ or $x = \gamma/\delta$ (vertical predator nullcline)

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# Cell 6.1 — Lotka-Volterra ODE
def lotka_volterra(t, y, alpha: float, beta: float, gamma: float, delta: float):
    """
    RHS of the Lotka-Volterra predator-prey system.

    Parameters
    ----------
    y : [x, y] — prey and predator populations
    alpha : prey birth rate
    beta  : predation rate
    gamma : predator death rate
    delta : predator growth efficiency

    Returns
    -------
    [dx/dt, dy/dt]
    """
    x, y_pred = y
    dxdt = alpha * x - beta * x * y_pred
    dydt = delta * x * y_pred - gamma * y_pred
    return [dxdt, dydt]

# Standard parameters (Strogatz example)
alpha, beta, gamma, delta = 1.0, 0.1, 1.5, 0.075

# Fixed points
x_fp = gamma / delta
y_fp = alpha / beta
print(f"Coexistence fixed point: (x*, y*) = ({x_fp:.2f}, {y_fp:.2f})")
print(f"Period estimate: T ≈ {2 * np.pi / np.sqrt(alpha * gamma):.2f} time units")

In [ ]:
# Cell 6.2 — Solve with scipy and with hand-rolled RK4
params = (alpha, beta, gamma, delta)
y0 = [10.0, 5.0]   # prey=10, predator=5
t_span = (0, 50)
t_eval = np.linspace(0, 50, 2000)

# scipy solve_ivp
sol = solve_ivp(lotka_volterra, t_span, y0, args=params,
                t_eval=t_eval, rtol=1e-8, atol=1e-10)
prey_scipy = sol.y[0]
pred_scipy = sol.y[1]

# RK4 from scratch
def rk4(f, y0, t_span, h):
    t0, T = t_span
    y0 = np.asarray(y0, dtype=float)
    n = int(np.ceil((T - t0) / h))
    t_arr = np.linspace(t0, t0 + n*h, n+1)
    y_arr = np.empty((n+1, len(y0)))
    y_arr[0] = y0
    for i in range(n):
        t_i, y_i = t_arr[i], y_arr[i]
        k1 = np.asarray(f(t_i,          y_i            ))
        k2 = np.asarray(f(t_i + h/2,    y_i + h/2 * k1))
        k3 = np.asarray(f(t_i + h/2,    y_i + h/2 * k2))
        k4 = np.asarray(f(t_i + h,      y_i + h   * k3))
        y_arr[i+1] = y_i + (h/6) * (k1 + 2*k2 + 2*k3 + k4)
    return t_arr, y_arr

f_lv = lambda t, y: lotka_volterra(t, y, *params)
t_rk4, y_rk4 = rk4(f_lv, y0, t_span, h=0.05)

max_diff = np.max(np.abs(y_rk4[-len(t_eval):, 0] - prey_scipy))
print(f"Max |RK4 - scipy| in prey trajectory: {max_diff:.2e}")

In [ ]:
# Cell 6.3 — Conservation law: V should be constant along trajectory
def V(x, y, alpha, beta, gamma, delta):
    """Lyapunov function — constant along Lotka-Volterra trajectories."""
    return delta*x - gamma*np.log(x) + beta*y - alpha*np.log(y)

V_traj = V(prey_scipy, pred_scipy, alpha, beta, gamma, delta)
V_range = V_traj.max() - V_traj.min()
print(f"V range along trajectory: {V_range:.2e}  (should be ~0 for exact solution)")
print(f"V std deviation: {V_traj.std():.2e}")
print(f"Conservation law satisfied: {'yes' if V_range < 1e-4 else 'no (increase rtol)'}")

In [ ]:
# Cell 6.4 — Parameter sensitivity: what happens when alpha doubles?
initial_states = [[10, 5], [20, 5], [10, 2]]
alphas = [0.5, 1.0, 2.0]

print("Parameter sensitivity — period estimate vs. alpha:")
for a in alphas:
    T_est = 2 * np.pi / np.sqrt(a * gamma)
    print(f"  alpha={a:.1f}  →  T ≈ {T_est:.2f}")

---
## Step 7 — Visualization (Portfolio Figure)

In [ ]:
# Cell 7.1 — Portfolio figure: time series + phase portrait + nullclines
fig = plt.figure(figsize=(15, 5))
gs = fig.add_gridspec(1, 3, wspace=0.35)

# Panel 1: Time series
ax1 = fig.add_subplot(gs[0])
ax1.plot(t_eval, prey_scipy, color="steelblue", lw=2, label="Prey ($x$)")
ax1.plot(t_eval, pred_scipy, color="tomato", lw=2, label="Predator ($y$)")
ax1.axhline(x_fp, color="steelblue", ls=":", alpha=0.5)
ax1.axhline(y_fp, color="tomato", ls=":", alpha=0.5)
ax1.set_xlabel("Time"); ax1.set_ylabel("Population size")
ax1.set_title("Lotka-Volterra: time series")
ax1.legend(frameon=False)

# Panel 2: Phase portrait with multiple ICs
ax2 = fig.add_subplot(gs[1])
for x0, y0_ic in [(10, 5), (15, 3), (6, 8)]:
    sol_ic = solve_ivp(lotka_volterra, (0, 50), [x0, y0_ic], args=params,
                       t_eval=np.linspace(0, 50, 2000), rtol=1e-8)
    ax2.plot(sol_ic.y[0], sol_ic.y[1], lw=1.5, alpha=0.85)
ax2.scatter([x_fp], [y_fp], s=100, color="black", zorder=10, label=f"Fixed point ({x_fp:.0f}, {y_fp:.0f})")
# Nullclines
x_nc = np.linspace(0.1, 35, 200)
ax2.axhline(alpha/beta, color="steelblue", ls="--", lw=1.2, label="Prey nullcline ($y=α/β$)")
ax2.axvline(gamma/delta, color="tomato", ls="--", lw=1.2, label="Pred. nullcline ($x=γ/δ$)")
ax2.set_xlabel("Prey ($x$)"); ax2.set_ylabel("Predator ($y$)")
ax2.set_title("Phase portrait")
ax2.legend(fontsize=7, loc="upper right")
ax2.set_xlim(0, 35); ax2.set_ylim(0, 25)

# Panel 3: Quiver field
ax3 = fig.add_subplot(gs[2])
x_q = np.linspace(0.5, 30, 18)
y_q = np.linspace(0.5, 22, 18)
X, Y = np.meshgrid(x_q, y_q)
U = alpha*X - beta*X*Y
Z_v = delta*X*Y - gamma*Y
speed = np.sqrt(U**2 + Z_v**2)
ax3.quiver(X, Y, U/speed, Z_v/speed, speed, cmap="viridis", alpha=0.75)
ax3.scatter([x_fp], [y_fp], s=100, color="red", zorder=10)
ax3.axhline(alpha/beta, color="steelblue", ls="--", lw=1)
ax3.axvline(gamma/delta, color="tomato", ls="--", lw=1)
ax3.set_xlabel("Prey"); ax3.set_ylabel("Predator")
ax3.set_title("Vector field")

plt.suptitle(
    "Lotka-Volterra predator-prey model\n"
    f"α={alpha}, β={beta}, γ={gamma}, δ={delta}",
    fontsize=11
)

# Save portfolio figure
portfolio_dir = Path("../../../../portfolio")
portfolio_dir.mkdir(exist_ok=True)
fig_path = portfolio_dir / "module02_lotka_volterra_phase_portrait.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
print(f"Portfolio figure saved: {fig_path}")

plt.show()

---
## Step 8 — Exercises

1. Derive the coexistence fixed point $(\gamma/\delta, \alpha/\beta)$ algebraically:
   set both equations to zero and solve. Show every step.
2. Verify the conservation law $V(x,y) = \delta x - \gamma \ln x + \beta y - \alpha \ln y$
   by differentiating $dV/dt$ along trajectories and showing it equals zero.
3. What happens to the oscillation period if $\alpha$ doubles? Predict from the
   period formula, then verify by running `solve_ivp` with the new parameter.
4. Add a carrying capacity to the prey equation: $dx/dt = \alpha x(1 - x/K) - \beta xy$.
   Solve numerically and compare the phase portrait to the classic Lotka-Volterra.
   Does the fixed point change? Do trajectories still close?

---
## Quiz — Active Recall

1. Write the Lotka-Volterra system from memory. Define every parameter.
2. What are the two fixed points? Which is biologically meaningful?
3. What are nullclines? What does it mean when a trajectory crosses a nullcline?
4. What is a conservation law? Why does it mean trajectories are closed loops?
5. In 30 seconds: explain to a non-specialist why lynx and hare populations oscillate.

---
## Papers Referenced

- Lotka, A.J. (1925). *Elements of Physical Biology*. Williams & Wilkins.
- Volterra, V. (1926). Fluctuations in the abundance of a species considered mathematically.
  *Nature*, 118, 558–560.

---
## Reflection

**Date completed:** ____________________

**Portfolio figure generated:** `portfolio/module02_lotka_volterra_phase_portrait.png`  
**Oral exam self-test:** Can you explain the full figure — equations, fixed points, nullclines,
conservation law — in under 3 minutes with no notes? [ ] Yes [ ] Not yet

---
*Next: `10_discrete_math_graphs_combinatorics.ipynb`*